# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset published: {metadata.datePublished}\nLicense: {metadata.license}\nVersion: {metadata.version}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Using `mlcroissant`, enumerate the record sets defined in the dataset schema. Each record set and field is referenced using its `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets())

for record_set in record_sets:
    print(f"Record Set ID: {record_set['@id']}")
    print(f"Record Set Name: {record_set.get('name', 'N/A')}")
    fields = record_set.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field.get('@id')} ({field.get('name', 'N/A')})")
        else:
            print(f"  - {field}")
    print()

### Example: Preview First Records
Print the first few records from a chosen record set (by `@id`).

In [ ]:
# Choose the main survey record set by @id (replace with correct id after inspection)
# For example, suppose the main survey record set id is 'cr:SurveyRecords'
main_record_set_id = None
for rs in record_sets:
    if 'mental' in (rs.get('name', '').lower() + rs.get('@id', '').lower()) or 'survey' in (rs.get('name', '').lower() + rs.get('@id', '').lower()):
        main_record_set_id = rs['@id']
        break
if not main_record_set_id:
    main_record_set_id = record_sets[0]['@id']  # fallback

for idx, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(f"Record {idx + 1}: {record}")
    if idx >= 2:
        break

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. Use the `@id` references from the overview section.

In [ ]:
# Build DataFrames for all record sets
dataframes = {}
for record_set in record_sets:
    rs_id = record_set['@id']
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"DataFrame for Record Set [{rs_id}]: Columns -> {df.columns.tolist()}")

# Preview the head of main record set
print(f"\nPreview of Data: {main_record_set_id}")
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric mental health indicator (e.g., GAD-7 score) by its field `@id` and demonstrate filtering and normalization. All references use the `@id`.

In [ ]:
# Identify available numeric fields by @id
df_main = dataframes[main_record_set_id]

numeric_field_id = None
for col in df_main.columns:
    if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback: choose first numeric column
    for col in df_main.columns:
        if pd.api.types.is_numeric_dtype(df_main[col]):
            numeric_field_id = col
            break

print(f"Using numeric field: {numeric_field_id}")

# Filtering: Example threshold for GAD-7 >= 10 (moderate anxiety)
threshold = 10
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by a demographic field (e.g., level_of_education)
group_field_id = None
for col in df_main.columns:
    if 'education' in col.lower():
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot the distribution of GAD-7 scores and their variation by education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of numeric mental health indicator scores
plt.figure(figsize=(8,4))
sns.histplot(df_main[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id} Scores")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field_id exists, plot boxplot by group
if group_field_id:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df_main[group_field_id], y=df_main[numeric_field_id])
    plt.title(f"{numeric_field_id} Scores by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset from Kilifi County, Kenya demonstrates AI-ready standards for African data infrastructure.
- Data exploration revealed distributions of mental health assessment scores and their relationships with demographic factors such as education.
- The dataset is structured for interoperability and reproducible research, with fields and record sets referenced by unique `@id`.